# Helper functions - set paths


In [1]:
# --- SETUP: shared paths + helpers (run once) ---
from pathlib import Path
import numpy as np
import nibabel as nib
from functools import lru_cache

SRC_ROOT     = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data/test_hires")
SRC_T1_DIR   = SRC_ROOT / "t1"
SRC_MASK_DIR = SRC_ROOT / "masks"
if not (SRC_T1_DIR.exists() and SRC_MASK_DIR.exists()):
    raise FileNotFoundError("Expected t1/ and masks/ subfolders under test_hires")

OUT_ROOT      = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Downsampled_Data")
OUT_ROOT.mkdir(parents=True, exist_ok=True)
JITTER_OUT     = OUT_ROOT / "test_hires_motion_slicejitter"
JITTER_T1_DIR  = JITTER_OUT / "t1"
JITTER_MASK_DIR= JITTER_OUT / "masks"
for d in (JITTER_OUT, JITTER_T1_DIR, JITTER_MASK_DIR):
    d.mkdir(parents=True, exist_ok=True)

def is_mask_name(name: str) -> bool:
    n = name.lower()
    return ("mask" in n) or ("lesion" in n)

def strip_ext(name: str) -> str:
    return name[:-7] if name.endswith(".nii.gz") else Path(name).stem

def norm_key(name: str) -> str:
    stem = strip_ext(name)
    for suf in ["_T1w_MNI_norm","_T1w_MNI","_T1w_brain","_T1w","_T1","_image","_img","_img_prepped"]:
        if stem.endswith(suf):
            stem = stem[: -len(suf)]
            break
    for suf in ["_lesion_mask_MNI_clean","_lesion_mask_MNI","_lesion_mask","_desc-lesion_mask","_mask","_mask_prepped"]:
        if stem.endswith(suf):
            stem = stem[: -len(suf)]
            break
    return stem.rstrip("_")

def _is_top_level(p: Path) -> bool:
    try:
        parent = p.parent.resolve()
    except Exception:
        return False
    return parent in {SRC_T1_DIR.resolve(), SRC_MASK_DIR.resolve()}

def _pair_maps(img_dir: Path, mask_dir: Path):
    imgs, msks = {}, {}
    for p in img_dir.rglob("*.nii.gz"):
        if is_mask_name(p.name):
            continue
        key = norm_key(p.name)
        keep = imgs.get(key)
        if key and (keep is None or (_is_top_level(p) and not _is_top_level(keep))):
            imgs[key] = p
    for p in mask_dir.rglob("*.nii.gz"):
        if not is_mask_name(p.name):
            continue
        key = norm_key(p.name)
        keep = msks.get(key)
        if key and (keep is None or (_is_top_level(p) and not _is_top_level(keep))):
            msks[key] = p
    keys = sorted(set(imgs) & set(msks))
    return {k: {"img": imgs[k], "msk": msks[k]} for k in keys}

def discover_pairs(img_dir: Path, mask_dir: Path):
    return [(v["img"], v["msk"]) for v in _pair_maps(img_dir, mask_dir).values()]

# basic I/O helpers reused below
@lru_cache(maxsize=128)
def load_nii(path: Path | str) -> nib.Nifti1Image:
    return nib.load(str(path))

def data_f32(img: nib.Nifti1Image) -> np.ndarray:
    return np.asarray(img.get_fdata(dtype=np.float32), dtype=np.float32)

def save_like(ref_img: nib.Nifti1Image, array: np.ndarray, out_path: Path, dtype=None):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = array.astype(dtype or np.float32)
    nib.save(nib.Nifti1Image(arr, ref_img.affine, ref_img.header.copy()), str(out_path))

def pad_or_crop_to(arr: np.ndarray, target_shape: tuple[int, int, int]) -> np.ndarray:
    out = arr
    for axis, tgt in enumerate(target_shape):
        cur = out.shape[axis]
        if cur == tgt:
            continue
        if cur > tgt:
            start = (cur - tgt) // 2
            sl = [slice(None)] * out.ndim
            sl[axis] = slice(start, start + tgt)
            out = out[tuple(sl)]
        else:
            pad_before = (tgt - cur) // 2
            pad_after = tgt - cur - pad_before
            pads = [(0, 0)] * out.ndim
            pads[axis] = (pad_before, pad_after)
            out = np.pad(out, pads, mode="edge")
    return out.astype(arr.dtype, copy=False)

# Preview pairing status up front
img_candidates  = [p for p in SRC_T1_DIR.rglob("*.nii.gz")]
mask_candidates = [p for p in SRC_MASK_DIR.rglob("*.nii.gz")]
_pair_preview   = _pair_maps(SRC_T1_DIR, SRC_MASK_DIR)
print(
    f"Found pairs: {len(_pair_preview)}  (imgs={len(img_candidates)}, masks={len(mask_candidates)})"
)
# (Viewer moved to Cell 7; helpers above are now shared by all cells.)

Found pairs: 214  (imgs=214, masks=214)


# 5) Slice-wise motion (rigid jitter)

Why / real-world: Patient motion during 2D acquisitions → slice-to-slice misalignment/blur.

What the code does

For each slice, applies a small random rotation (±a few degrees) and pixel shift (±a few px):
rotate(..., order=1) for image, order=0 for mask; then shift(...) similarly.

This produces slice-to-slice misalignments and slight blurring/ghosting from interpolation.

Voxel spacing unchanged; just geometry perturbations.

What this mimics

2D multi-slice acquisitions where the patient moves between slice excitations → slice stack doesn’t line up perfectly.

Very common in restless patients, pediatrics, or longer scans.

Why it’s useful

Motion is one of the biggest real-world degraders. Even tiny rotations/shift destroy fine boundaries and create zebra-like slice seams.

Caveats

Real motion can be continuous and within-TR; this is a discrete per-slice model (captures the dominant visual effect).

In [2]:
# === 5) SLICE-WISE MOTION JITTER: small per-slice rotations/shifts (XY plane), Z intact ===
from pathlib import Path
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate, shift
import shutil

OUT_DIR     = JITTER_OUT
OUT_IMG_DIR = JITTER_T1_DIR
OUT_MSK_DIR = JITTER_MASK_DIR
deg_range  = 2.0
px_range   = 2.0
SEED       = 7
OVERWRITE  = False

OUT_DIR.mkdir(parents=True, exist_ok=True)

pairs = discover_pairs(SRC_T1_DIR, SRC_MASK_DIR)
print(f"Discovered unique pairs: {len(pairs)} (deduped across flat + subfolders)")

rng   = np.random.default_rng(SEED)
wrote = 0

for i, (img_p, msk_p) in enumerate(pairs, 1):
    img_ref = load_nii(img_p)
    x = data_f32(img_ref)
    H, W, Z = x.shape
    xm = np.empty_like(x, dtype=np.float32)

    angs = rng.uniform(-deg_range, deg_range, size=Z)
    dxs  = rng.uniform(-px_range,  px_range,  size=Z)
    dys  = rng.uniform(-px_range,  px_range,  size=Z)

    for k in range(Z):
        ang, dx, dy = float(angs[k]), float(dxs[k]), float(dys[k])
        sl = rotate(x[:, :, k], angle=ang, reshape=False, order=1, mode="nearest")
        sl = shift (sl,        shift=(dy, dx),      order=1, mode="nearest")
        xm[:, :, k] = sl

    base = strip_ext(img_p.name).replace("_T1w_MNI_norm","").replace("_T1w","")
    out_img = OUT_IMG_DIR / f"{base}_T1w_MNI_norm.nii.gz"
    out_msk = OUT_MSK_DIR / f"{base}_lesion_mask_MNI_clean.nii.gz"

    if not OVERWRITE and out_img.exists() and out_msk.exists():
        if i % 25 == 0 or i == len(pairs):
            print(f"[{i}/{len(pairs)}] (skip, exists) {out_img.name}")
        continue

    save_like(img_ref, xm, out_img, dtype=np.float32)
    out_msk.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(msk_p), str(out_msk))
    wrote += 1

    if i % 10 == 0 or i == len(pairs):
        print(f"[{i}/{len(pairs)}] wrote {out_img.name} & {out_msk.name}")

print(f"Done → {OUT_DIR} | wrote {wrote} case(s)")


Discovered unique pairs: 214 (deduped across flat + subfolders)
[10/214] wrote sub-M2074_ses-2183_T1w_MNI_norm.nii.gz & sub-M2074_ses-2183_lesion_mask_MNI_clean.nii.gz
[10/214] wrote sub-M2074_ses-2183_T1w_MNI_norm.nii.gz & sub-M2074_ses-2183_lesion_mask_MNI_clean.nii.gz
[20/214] wrote sub-M2100_ses-6369_T1w_MNI_norm.nii.gz & sub-M2100_ses-6369_lesion_mask_MNI_clean.nii.gz
[20/214] wrote sub-M2100_ses-6369_T1w_MNI_norm.nii.gz & sub-M2100_ses-6369_lesion_mask_MNI_clean.nii.gz
[30/214] wrote sub-M2140_ses-631_T1w_MNI_norm.nii.gz & sub-M2140_ses-631_lesion_mask_MNI_clean.nii.gz
[30/214] wrote sub-M2140_ses-631_T1w_MNI_norm.nii.gz & sub-M2140_ses-631_lesion_mask_MNI_clean.nii.gz
[40/214] wrote sub-M2175_ses-703_T1w_MNI_norm.nii.gz & sub-M2175_ses-703_lesion_mask_MNI_clean.nii.gz
[40/214] wrote sub-M2175_ses-703_T1w_MNI_norm.nii.gz & sub-M2175_ses-703_lesion_mask_MNI_clean.nii.gz
[50/214] wrote sub-M2220_ses-359_T1w_MNI_norm.nii.gz & sub-M2220_ses-359_lesion_mask_MNI_clean.nii.gz
[50/214] w

In [ ]:
import numpy as np, nibabel as nib, matplotlib.pyplot as plt, ipywidgets as W
from functools import lru_cache
from IPython.display import display, clear_output

orig_pairs = _pair_maps(SRC_T1_DIR, SRC_MASK_DIR)
ds_pairs   = _pair_maps(JITTER_T1_DIR, JITTER_MASK_DIR)
common_keys = sorted(set(orig_pairs) & set(ds_pairs))
if not common_keys:
    raise RuntimeError("No overlapping cases between original and motion-jitter datasets.")

@lru_cache(maxsize=64)
def _load_vol(path: str):
    img = nib.load(path)
    data = img.get_fdata().astype(np.float32)
    return data[..., 0] if data.ndim == 4 and data.shape[-1] == 1 else data

def _normalize(vol):
    nz = vol[vol > 0]
    if nz.size == 0:
        return vol * 0
    p1, p99 = np.percentile(nz, [1, 99])
    return np.clip((vol - p1) / max(p99 - p1, 1e-5), 0, 1)

key_dd   = W.Dropdown(options=common_keys, description="Case:", layout=W.Layout(width="45%"))
slice_sl = W.IntSlider(description="Slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="45%"))
out = W.Output()

def _update_slider(*_):
    vol = _load_vol(str(orig_pairs[key_dd.value]["img"]))
    slice_sl.max = max(0, vol.shape[2] - 1)

def _render(*_):
    with out:
        clear_output(wait=True)
        key = key_dd.value
        img_orig = _load_vol(str(orig_pairs[key]["img"]))
        msk_orig = (_load_vol(str(orig_pairs[key]["msk"])) > 0.5)
        img_ds   = _load_vol(str(ds_pairs[key]["img"]))
        msk_ds   = (_load_vol(str(ds_pairs[key]["msk"])) > 0.5)
        z = int(slice_sl.value)

        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        titles = ["Original", "Slice-wise motion jitter"]
        for ax, img, msk, title in zip(axes,
                                       (_normalize(img_orig[..., z]), _normalize(img_ds[..., z])),
                                       (msk_orig[..., z], msk_ds[..., z]),
                                       titles):
            ax.imshow(img.T, cmap="gray", origin="lower")
            ax.contour(msk.T, levels=[0.5], colors="r", linewidths=0.8)
            ax.set_title(f"{title}\n{key} | slice {z}")
            ax.axis("off")
        plt.tight_layout(); plt.show()

_update_slider()
_render()
key_dd.observe(_update_slider, names="value")
key_dd.observe(_render, names="value")
slice_sl.observe(_render, names="value")
display(W.VBox([W.HBox([key_dd, slice_sl]), out]))